In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Conv1D, BatchNormalization, Bidirectional, LSTM, Concatenate, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber, MeanSquaredError, LogCosh, MeanAbsoluteError
from tensorflow.keras.initializers import GlorotUniform, Orthogonal
import json
from scipy.stats import zscore
from utils import random_extraction, extract_hrv_features, fast_predict, evaluate_imputation_performance

I0000 00:00:1782504358.198182 1678573 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782504360.321890 1678573 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Cargar los datos
series_list = 'series/'
files = ['006.txt',
        '16265.txt',
        '16273.txt',
        '16786.txt',
        '16795.txt',
        '17453.txt',
        '18177.txt',
        '18184.txt',
        '16539.txt',
        'nsr054RRcl.txt',
        '005.txt',
        '16420.txt',
        '19140.txt',
        'nsr047RRcl.txt',
        'nsr048RRcl.txt',
        'nsr049RRcl.txt',
        '008.txt',
        '009.txt',
        '013.txt']
window_size = 20
np.random.seed(42)
SEED = 53
# select 50 random files
files_train = np.random.choice(files, size=11, replace=False) # np.array(['16795.txt', '013.txt', '18177.txt', 'nsr047RRcl.txt', '006.txt', '16786.txt', '19140.txt', 'nsr054RRcl.txt', '16420.txt', 'nsr048RRcl.txt'])

# the rest of the files will be used for testing
files_test = [f for f in files if f not in files_train]

feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50', 'porta', 'std']
rr_cols = [f'rr_{i}' for i in range(1, 21)]
rr_last_col = 'rr_20'

# --- 2. ESCALADO Y RESHAPE ---
scaler_feats = StandardScaler()
original_serie = np.loadtxt(os.path.join(series_list, files_test[0]), dtype=int).astype(float)

In [3]:
# 1. Define paths
save_dir = ''
model_path = os.path.join(save_dir, 'cnn_lstm_hybrid.keras')
json_path = os.path.join(save_dir, 'scalers_params.json')

# 2. Load the Keras Model
print("Loading model...")
loaded_model = load_model(model_path)
print("✅ Model loaded successfully.")

# 3. Load the Scaler Parameters
print("Loading scalers...")
with open(json_path, 'r') as json_file:
    loaded_scalers = json.load(json_file)

# 4. Extract parameters back into Numpy arrays (using float32 for Keras compatibility)
# Sequence Scaler (Global Mean/Scale)
seq_mean = np.array(loaded_scalers['scaler_seq']['mean'], dtype=np.float32)
seq_scale = np.array(loaded_scalers['scaler_seq']['scale'], dtype=np.float32)

# Features Scaler (8 features)
feats_mean = np.array(loaded_scalers['scaler_feats']['mean'], dtype=np.float32)
feats_scale = np.array(loaded_scalers['scaler_feats']['scale'], dtype=np.float32)

# Target Scaler (Delta RR)
y_mean = np.float32(loaded_scalers['y_scaler']['mean'][0])
y_scale = np.float32(loaded_scalers['y_scaler']['scale'][0])
print("✅ Scaler parameters reconstructed.")

Loading model...


E0000 00:00:1782504375.631819 1678573 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1782504375.632153 1679023 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1782504375.644106 1678573 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W0000 00:00:1782504375.644726 1678573 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


✅ Model loaded successfully.
Loading scalers...
✅ Scaler parameters reconstructed.


In [ ]:
# Create your array of percentages
percents_array = np.arange(0.005, 0.8, 0.005)

# Run the function
rmse_results, mae_results, r2_results, corr_results = evaluate_imputation_performance(
    original_serie=original_serie,
    percents_to_eliminate=percents_array,
    loaded_model=loaded_model,
    feats_mean=feats_mean, feats_scale=feats_scale,
    seq_mean=seq_mean, seq_scale=seq_scale,
    y_mean=y_mean, y_scale=y_scale,
    feature_cols=feature_cols, rr_cols=rr_cols
)

# You can now plot these arrays directly against percents_array!

Starting autoregressive imputation across 159 thresholds...
Evaluating elimination: 0.50%
Evaluating elimination: 1.00%
Evaluating elimination: 1.50%
Evaluating elimination: 2.00%
Evaluating elimination: 2.50%
Evaluating elimination: 3.00%
Evaluating elimination: 3.50%
Evaluating elimination: 4.00%
Evaluating elimination: 4.50%
Evaluating elimination: 5.00%
Evaluating elimination: 5.50%
Evaluating elimination: 6.00%
Evaluating elimination: 6.50%
Evaluating elimination: 7.00%
Evaluating elimination: 7.50%
Evaluating elimination: 8.00%
Evaluating elimination: 8.50%
Evaluating elimination: 9.00%
Evaluating elimination: 9.50%
Evaluating elimination: 10.00%
Evaluating elimination: 10.50%
Evaluating elimination: 11.00%
Evaluating elimination: 11.50%
Evaluating elimination: 12.00%
Evaluating elimination: 12.50%
Evaluating elimination: 13.00%
Evaluating elimination: 13.50%
Evaluating elimination: 14.00%
Evaluating elimination: 14.50%
Evaluating elimination: 15.00%
Evaluating elimination: 15.50

In [ ]:
# 1. Extract a clean name for the subject (e.g., removes '.txt' from 'patient_01.txt')
subject_name = os.path.splitext(files_test[0])[0]

# 2. Combine the arrays into a structured Pandas DataFrame
# (Assuming 'percents_array' is the array you used for the loop)
results_df = pd.DataFrame({
    'Percent_Eliminated': percents_array,
    'RMSE': rmse_results,
    'MAE': mae_results,
    'R2': r2_results,
    'Correlation': corr_results
})

# 3. Define where you want to save it
results_dir = ''
os.makedirs(results_dir, exist_ok=True) # Creates the folder if it doesn't exist

save_path = os.path.join(results_dir, f'imputation_metrics_{subject_name}.csv')

# 4. Save as CSV (index=False prevents Pandas from saving the row numbers)
results_df.to_csv(save_path, index=False)

print(f"✅ Results successfully saved to: {save_path}")